# Python revision extraction

Extracts the **code snippets** (`PostBlockTypeId = 2`) of every accepted, revised Python
answer in SOTorrent, together with **all of their revisions**, and writes each one out as
its own file — the Python analog of the Java `writeDiffFiles` stage
(`SOTorrentAnalyzer/PostBlockProcessor.java`) whose output lives in `../../java_files/`.

**Input:** `files/acceptedWithVersionAnswer_python.txt` — one accepted-answer `PostId` per
line (built by `extract_answer_list.py`; see `README.md`).

**Output:** `../../python_files/{PostId}/` — one folder per answer, containing:

| File | Contents |
|---|---|
| `{PostId}_{LocalId}_{PostHistoryId}.py` | The code block's content in that specific revision. One file per row in `PostBlockVersion`. |
| `{PostId}_{LocalId}_recent.py` | Same content as the row where `MostRecentVersion = 1`. |
| `{PostId}_{LocalId}_original.py` | Content of the block's *root* version — `PostBlockVersion.RootPostBlockVersionId` — i.e. the earliest ancestor in the current unbroken edit chain (not necessarily the post's very first revision, since a chain can break and restart). |

This mirrors the example at `../../java_files/2793153/` (verified against the live
`sotorrent` DB — e.g. `2793153_10_original.java`'s content matches the
`PostBlockVersion` row at its `RootPostHistoryId`, and `2793153_10_recent.java` matches
the row with `MostRecentVersion = 1`).

`LocalId` identifies a block's position within one particular revision (`PostHistoryId`);
the same logical snippet keeps the same `LocalId` across the revisions where it exists,
which is why every revision file below is grouped by `(PostId, LocalId)`.

Run the cells in order. The **Configuration** cell controls DB connection, paths, batch
size, and the file extension (`.py`, per the requested naming convention). The full
305k-post run is gated behind `RUN_FULL_EXTRACTION` — try the **Sanity check** cell on a
handful of posts first.

## Configuration

In [1]:
import time
from itertools import groupby
from operator import itemgetter
from pathlib import Path

import mysql.connector

# This notebook lives in <repo>/analysis/. The Java stage's outputs (../java_files/) and
# this notebook's outputs (../python_files/) both live one level above the repo root.
REPO_ROOT = Path.cwd().resolve().parent
STUDY_ROOT = REPO_ROOT.parent

ANSWER_LIST = REPO_ROOT / "files" / "acceptedWithVersionAnswer_python.txt"
OUTPUT_DIR = STUDY_ROOT / "python_files"

DB_CONFIG = dict(host="127.0.0.1", user="root", password="", database="sotorrent")

EXTENSION = "py"          # {PostID}_{LocalID}_{revisionID}.py
BATCH_SIZE = 200          # posts per SQL round-trip
PROGRESS_EVERY = 2_000    # posts, for the full run's progress log

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Answer list: {ANSWER_LIST}")
print(f"Output dir:  {OUTPUT_DIR}")


Answer list: /Users/chaiyong/Downloads/do_not_delete/Matcha_Study/SOPostProcessor/files/acceptedWithVersionAnswer_python.txt
Output dir:  /Users/chaiyong/Downloads/do_not_delete/Matcha_Study/python_files


## Helpers

- `load_post_ids` reads the bare-integer answer-id file.
- `fetch_batch` pulls every code-block revision (`PostBlockTypeId = 2`) for a batch of
  `PostId`s in one query — far fewer round-trips than the Java stage's one-query-per-post
  loop.
- `process_rows` groups by `(PostId, LocalId)` and writes the per-revision, `_original`,
  and `_recent` files described above. Because a batch already contains a post's *entire*
  code-block history, a root row's content is always available locally via
  `id_to_row` — no extra query needed to resolve `_original`.

In [2]:
def load_post_ids(path: Path, limit: int | None = None) -> list[int]:
    ids = [int(line) for line in path.read_text().split() if line.strip()]
    return ids[:limit] if limit is not None else ids


def connect():
    return mysql.connector.connect(**DB_CONFIG)


def fetch_batch(cursor, post_ids: list[int]):
    placeholders = ",".join(["%s"] * len(post_ids))
    query = f"""
        SELECT Id, PostId, PostHistoryId, LocalId, MostRecentVersion,
               RootPostBlockVersionId, Content
        FROM PostBlockVersion
        WHERE PostBlockTypeId = 2 AND PostId IN ({placeholders})
        ORDER BY PostId, LocalId, PostHistoryId
    """
    cursor.execute(query, post_ids)
    return cursor.fetchall()


def write_file(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def process_rows(rows, output_dir: Path, extension: str) -> tuple[int, set[int]]:
    """Write one file per code-block revision plus _original/_recent per (PostId, LocalId)."""
    id_to_row = {row[0]: row for row in rows}
    files_written = 0
    posts_seen: set[int] = set()

    for (post_id, local_id), group in groupby(rows, key=itemgetter(1, 3)):
        group = list(group)
        posts_seen.add(post_id)
        post_dir = output_dir / str(post_id)
        recent_row = None

        for row in group:
            _, _, post_history_id, _, most_recent_version, _, content = row
            write_file(post_dir / f"{post_id}_{local_id}_{post_history_id}.{extension}", content)
            files_written += 1
            if most_recent_version == 1:
                recent_row = row

        if recent_row is not None:
            _, _, _, _, _, root_id, recent_content = recent_row
            write_file(post_dir / f"{post_id}_{local_id}_recent.{extension}", recent_content)
            files_written += 1

            original_row = id_to_row.get(root_id)
            if original_row is not None:
                write_file(post_dir / f"{post_id}_{local_id}_original.{extension}", original_row[6])
                files_written += 1

    return files_written, posts_seen


## Sanity check

Runs the extraction on a handful of posts and prints the resulting file tree, so the
naming convention and file contents can be eyeballed before committing to the full run.

In [3]:
sample_ids = load_post_ids(ANSWER_LIST, limit=5)
print("Sample post ids:", sample_ids)

db = connect()
try:
    cursor = db.cursor()
    rows = fetch_batch(cursor, sample_ids)
    print(f"Fetched {len(rows)} code-block revision rows")
    files_written, posts_seen = process_rows(rows, OUTPUT_DIR, EXTENSION)
    cursor.close()
finally:
    db.close()

print(f"Wrote {files_written} files across {len(posts_seen)} posts (of {len(sample_ids)} requested)")
for post_id in sample_ids:
    post_dir = OUTPUT_DIR / str(post_id)
    if post_dir.exists():
        names = sorted(p.name for p in post_dir.iterdir())
        print(f"\n{post_dir}/  ({len(names)} files)")
        for name in names[:10]:
            print(" ", name)
        if len(names) > 10:
            print(f"  ... and {len(names) - 10} more")
    else:
        print(f"\n{post_dir}/  -- no code blocks found for this post")


Sample post ids: [595, 1852, 1857, 2316, 2982]
Fetched 41 code-block revision rows
Wrote 71 files across 5 posts (of 5 requested)

/Users/chaiyong/Downloads/do_not_delete/Matcha_Study/python_files/595/  (23 files)
  595_10_129314947.py
  595_10_129380722.py
  595_10_original.py
  595_10_recent.py
  595_2_129314947.py
  595_2_129380722.py
  595_2_604.py
  595_2_original.py
  595_2_recent.py
  595_4_129314947.py
  ... and 13 more

/Users/chaiyong/Downloads/do_not_delete/Matcha_Study/python_files/1852/  (4 files)
  1852_2_39188.py
  1852_2_39189.py
  1852_2_original.py
  1852_2_recent.py

/Users/chaiyong/Downloads/do_not_delete/Matcha_Study/python_files/1857/  (6 files)
  1857_1_165462462.py
  1857_1_211903859.py
  1857_1_35095463.py
  1857_1_66191434.py
  1857_1_original.py
  1857_1_recent.py

/Users/chaiyong/Downloads/do_not_delete/Matcha_Study/python_files/2316/  (3 files)
  2316_2_118846228.py
  2316_2_original.py
  2316_2_recent.py

/Users/chaiyong/Downloads/do_not_delete/Matcha_Stud

## Full extraction

Processes every post in `acceptedWithVersionAnswer_python.txt` (~306k answers as of
2026-07-21). This writes a large number of files and can take a long time — it's gated
behind `RUN_FULL_EXTRACTION` so re-running the notebook top-to-bottom doesn't kick it off
by accident.

Resumable: a post is skipped if its output folder already exists and is non-empty, so an
interrupted run can just be restarted.

In [6]:
RUN_FULL_EXTRACTION = True  # flip to True to actually run the full extraction

if RUN_FULL_EXTRACTION:
    all_ids = load_post_ids(ANSWER_LIST)
    print(f"Loaded {len(all_ids):,} post ids")

    todo_ids = [pid for pid in all_ids if not (OUTPUT_DIR / str(pid)).exists()]
    skipped = len(all_ids) - len(todo_ids)
    if skipped:
        print(f"Skipping {skipped:,} posts already extracted; {len(todo_ids):,} remaining")

    db = connect()
    total_files = 0
    total_posts_with_code = 0
    t0 = time.perf_counter()
    try:
        cursor = db.cursor()
        for i in range(0, len(todo_ids), BATCH_SIZE):
            batch = todo_ids[i:i + BATCH_SIZE]
            rows = fetch_batch(cursor, batch)
            files_written, posts_seen = process_rows(rows, OUTPUT_DIR, EXTENSION)
            total_files += files_written
            total_posts_with_code += len(posts_seen)

            # posts with zero code blocks still count as "done" so they aren't retried
            for pid in batch:
                (OUTPUT_DIR / str(pid)).mkdir(parents=True, exist_ok=True)

            done = i + len(batch)
            if done % PROGRESS_EVERY < BATCH_SIZE:
                elapsed = time.perf_counter() - t0
                rate = done / elapsed if elapsed > 0 else 0
                print(f"  ...{done:,}/{len(todo_ids):,} posts "
                      f"({total_files:,} files written, {rate:.1f} posts/s)", end="\r", flush=True)
        cursor.close()
    finally:
        db.close()

    elapsed = time.perf_counter() - t0
    print(f"\nDone in {elapsed / 60:.1f} min. "
          f"{total_files:,} files written for {total_posts_with_code:,} posts with code blocks "
          f"(of {len(todo_ids):,} newly processed, {skipped:,} already done).")
else:
    print("RUN_FULL_EXTRACTION is False -- skipping. Flip it to True to run the full extraction.")


Loaded 305,768 post ids
Skipping 5 posts already extracted; 305,763 remaining
  ...304,000/305,763 posts (2,902,556 files written, 649.2 posts/s)
Done in 7.8 min. 2,915,855 files written for 305,762 posts with code blocks (of 305,763 newly processed, 5 already done).


## Summary

Quick stats over whatever has been written to `OUTPUT_DIR` so far.

In [5]:
post_dirs = [p for p in OUTPUT_DIR.iterdir() if p.is_dir()]
all_files = [f for p in post_dirs for f in p.iterdir()]
posts_with_code = [p for p in post_dirs if any(p.iterdir())]

print(f"Post folders:        {len(post_dirs):,}")
print(f"Posts with code:     {len(posts_with_code):,}")
print(f"Total files written: {len(all_files):,}")


Post folders:        5
Posts with code:     5
Total files written: 71
